# Séance 3 — Gradient, contours et segmentation classique

## Objectif du notebook
Ce notebook sert de **démo guidée** pour la séance 3.

On va construire pas à pas une chaîne simple :

**prétraitement → gradient / contours → segmentation**

## Ce que vous devez retenir
- Le **gradient** mesure des variations locales d'intensité.
- Les opérateurs **Roberts, Prewitt, Sobel** sont trois approximations discrètes des dérivées.
- **Canny** est une chaîne plus robuste que “gradient + seuil”.
- Un **contour** n'est pas une **segmentation**.
- Les méthodes de segmentation classiques les plus simples ici sont :
  - seuil fixe,
  - seuil automatique d'Otsu,
  - watershed avec marqueurs.

## Rappel très court
- **Séance 1** : image numérique, histogramme, acquisition, bruit.
- **Séance 2** : contraste, convolution, flou gaussien, filtrage.
- **Séance 3** : on ne veut plus seulement améliorer l'image ; on veut **extraire des structures**.

In [ ]:
#%pip install opencv-python

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

from skimage import data

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["axes.grid"] = False

## 1. Fonctions d'affichage
On commence par quelques fonctions utilitaires.  
Elles seront réutilisées tout au long du notebook.

In [ ]:
def show_image(img, title="Image", cmap="gray", figsize=(6, 4)):
    """Affiche une image unique.

    Paramètres
    ----------
    img : numpy.ndarray
        Image 2D ou 3D.
    title : str
        Titre de la figure.
    cmap : str
        Colormap pour une image 2D.
    figsize : tuple
        Taille de la figure.

    Retour
    ------
    None

    Remarques pédagogiques
    ----------------------
    - Une image en niveaux de gris se lit comme une matrice 2D.
    - Une image couleur OpenCV est souvent codée en BGR.
    - Ici, si l'image est 3D, on la convertit en RGB pour l'affichage.
    """
    plt.figure(figsize=figsize)
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
def show_images(images, titles, ncols=3, cmap="gray", figsize=(14, 8)):
    """Affiche plusieurs images dans une grille simple.

    Paramètres
    ----------
    images : list
        Liste d'images.
    titles : list
        Liste des titres associés.
    ncols : int
        Nombre de colonnes.
    cmap : str
        Colormap pour les images 2D.
    figsize : tuple
        Taille de la figure.

    Retour
    ------
    None

    Remarques pédagogiques
    ----------------------
    - Utile pour comparer plusieurs méthodes sur la même image.
    - La comparaison visuelle est centrale en traitement d'image.
    """
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap=cmap)
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### Démo rapide

In [ ]:
img_demo = data.camera()
show_image(img_demo, "Démo — image unique")
show_images(
    [img_demo, data.coins(), data.moon()],
    ["camera", "coins", "moon"],
    ncols=3,
    figsize=(14, 4)
)

## À observer
- Avant d'analyser, il faut d'abord **bien voir**.
- Une comparaison côte à côte fait gagner du temps et évite les jugements à l'aveugle.

## 2. Images de travail
On utilise :
- une image **synthétique** pour comprendre simplement ;
- des images **réelles** pour voir les limites des méthodes.

In [ ]:
img_camera = data.camera()
img_coins = data.coins()
img_moon = data.moon()

img_synth = np.zeros((256, 256), dtype=np.uint8)
cv2.rectangle(img_synth, (30, 30), (110, 180), 180, thickness=-1)
cv2.circle(img_synth, (180, 120), 45, 255, thickness=-1)
cv2.line(img_synth, (20, 220), (230, 220), 120, thickness=3)

show_images(
    [img_synth, img_camera, img_coins, img_moon],
    ["Image synthétique", "camera", "coins", "moon"],
    ncols=2,
    figsize=(12, 8)
)

## À observer
- L'image synthétique permet de comprendre les méthodes sans distraction.
- `camera` est utile pour les contours.
- `coins` est utile pour la segmentation d'objets.
- `moon` montre qu'une image réelle peut avoir des transitions plus douces.

## 3. Prétraitement : lisser avant de dériver
Rappel direct de la séance 2 :

le gradient amplifie volontiers le bruit.  
Donc, dans un pipeline réaliste, on lisse souvent **avant** de calculer un gradient.

In [ ]:
def gaussian_blur(image, ksize=5, sigma=1.0):
    """Applique un flou gaussien.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.
    ksize : int
        Taille du noyau gaussien, impaire.
    sigma : float
        Écart-type de la gaussienne.

    Retour
    ------
    numpy.ndarray
        Image lissée.

    Remarques pédagogiques
    ----------------------
    - Le gaussien réduit les variations locales parasites.
    - Il ne supprime pas le bruit parfaitement, mais il le calme.
    - Plus sigma est grand, plus l'image est lissée.
    """
    return cv2.GaussianBlur(image, (ksize, ksize), sigmaX=sigma)

### Démo immédiate

In [ ]:
rng = np.random.default_rng(42)
noise = rng.normal(0, 20, size=img_synth.shape)
img_synth_noisy = np.clip(img_synth.astype(np.float32) + noise, 0, 255).astype(np.uint8)
img_synth_blur = gaussian_blur(img_synth_noisy, ksize=5, sigma=1.2)

show_images(
    [img_synth, img_synth_noisy, img_synth_blur],
    ["Image propre", "Image bruitée", "Image bruitée + flou gaussien"],
    ncols=3,
    figsize=(15, 4)
)

## À observer
- Le bruit ajoute de petites variations partout.
- Le flou gaussien rend l'image plus régulière.
- On paie ce calme avec un léger flou sur les contours.

## 4. Gradient de Sobel
Le gradient mesure des **variations locales d'intensité**.

On va calculer :
- **Gx** : variation selon x,
- **Gy** : variation selon y,
- **magnitude** : force de variation,
- **orientation** : direction du gradient.

Ici, pour l'orientation, on utilise `atan2(Gy, Gx)` plutôt que `arctan(Gy/Gx)`.

In [ ]:
def compute_sobel(image):
    """Calcule Gx, Gy, magnitude et orientation avec Sobel.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.

    Retour
    ------
    gx : numpy.ndarray
        Dérivée selon x.
    gy : numpy.ndarray
        Dérivée selon y.
    magnitude : numpy.ndarray
        Norme du gradient.
    angle_deg : numpy.ndarray
        Orientation du gradient en degrés, dans [0, 180).

    Remarques pédagogiques
    ----------------------
    - Sobel approxime la dérivée discrète.
    - La magnitude met en évidence les frontières probables.
    - L'orientation du gradient est perpendiculaire au contour local.
    - On travaille en float pour éviter de casser l'information.
    """
    image_f = image.astype(np.float32)

    gx = cv2.Sobel(image_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(image_f, cv2.CV_32F, 0, 1, ksize=3)

    magnitude = np.sqrt(gx**2 + gy**2)
    angle_deg = (np.rad2deg(np.arctan2(gy, gx)) + 180) % 180

    return gx, gy, magnitude, angle_deg

### Démo immédiate

In [ ]:
gx, gy, mag, angle = compute_sobel(img_camera)

show_images(
    [
        img_camera,
        np.abs(gx),
        np.abs(gy),
        cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    ],
    ["Originale", "|Gx|", "|Gy|", "Magnitude Sobel"],
    ncols=4,
    figsize=(16, 4)
)

## À observer
- `Gx` réagit aux variations horizontales de l'intensité.
- `Gy` réagit aux variations verticales.
- La magnitude regroupe ces deux informations.
- Un fort gradient n'est pas encore un contour binaire : c'est une **carte continue**.

## 5. Visualiser l'orientation du gradient
L'orientation est utile pour comprendre la géométrie locale, et elle sera cruciale dans l'idée de Canny.

In [ ]:
def visualize_gradient_orientation(angle_deg):
    """Normalise l'orientation pour l'afficher comme une image.

    Paramètres
    ----------
    angle_deg : numpy.ndarray
        Orientation en degrés dans [0, 180).

    Retour
    ------
    numpy.ndarray
        Image uint8 prête à afficher.

    Remarques pédagogiques
    ----------------------
    - Ici, ce n'est pas une mesure physique supplémentaire.
    - C'est seulement une façon visuelle de voir les directions.
    """
    return np.uint8((angle_deg / 180.0) * 255)

### Démo immédiate

In [ ]:
angle_vis = visualize_gradient_orientation(angle)
show_images(
    [img_camera, angle_vis],
    ["Originale", "Orientation du gradient (visualisation)"],
    ncols=2,
    figsize=(12, 4)
)

## À observer
- Deux zones peuvent avoir une magnitude proche mais une orientation différente.
- Le gradient ne dit pas seulement “où ça change”, il dit aussi “dans quelle direction ça change”.

## 6. Roberts, Prewitt, Sobel
Trois opérateurs classiques pour approximer les dérivées.

- **Roberts** : compact, nerveux, sensible au bruit.
- **Prewitt** : simple, lisible.
- **Sobel** : léger lissage intégré, souvent le plus confortable.

In [ ]:
def compute_roberts(image):
    """Calcule la magnitude du gradient avec l'opérateur de Roberts.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.

    Retour
    ------
    gx : numpy.ndarray
        Réponse diagonale 1.
    gy : numpy.ndarray
        Réponse diagonale 2.
    magnitude : numpy.ndarray
        Norme du gradient.

    Remarques pédagogiques
    ----------------------
    - Roberts utilise un support 2x2.
    - Il est compact, mais souvent plus sensible au bruit.
    """
    image_f = image.astype(np.float32)

    kx = np.array([[1, 0],
                   [0, -1]], dtype=np.float32)
    ky = np.array([[0, 1],
                   [-1, 0]], dtype=np.float32)

    gx = cv2.filter2D(image_f, cv2.CV_32F, kx)
    gy = cv2.filter2D(image_f, cv2.CV_32F, ky)
    magnitude = np.sqrt(gx**2 + gy**2)

    return gx, gy, magnitude

In [ ]:
def compute_prewitt(image):
    """Calcule la magnitude du gradient avec l'opérateur de Prewitt.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.

    Retour
    ------
    gx : numpy.ndarray
        Dérivée selon x.
    gy : numpy.ndarray
        Dérivée selon y.
    magnitude : numpy.ndarray
        Norme du gradient.

    Remarques pédagogiques
    ----------------------
    - Prewitt fait une différence finie avec un lissage uniforme.
    - C'est un bon intermédiaire pour comprendre Sobel.
    """
    image_f = image.astype(np.float32)

    kx = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=np.float32)

    ky = np.array([[-1, -1, -1],
                   [ 0,  0,  0],
                   [ 1,  1,  1]], dtype=np.float32)

    gx = cv2.filter2D(image_f, cv2.CV_32F, kx)
    gy = cv2.filter2D(image_f, cv2.CV_32F, ky)
    magnitude = np.sqrt(gx**2 + gy**2)

    return gx, gy, magnitude

In [ ]:
def normalize_to_uint8(image):
    """Normalise une image réelle vers [0, 255] pour l'affichage.

    Paramètres
    ----------
    image : numpy.ndarray
        Image en float ou int.

    Retour
    ------
    numpy.ndarray
        Image uint8 normalisée.

    Remarques pédagogiques
    ----------------------
    - Beaucoup de sorties intermédiaires du traitement d'image
      ne sont pas directement affichables.
    - On normalise ici uniquement pour voir.
    """
    norm = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    return norm.astype(np.uint8)

### Démo immédiate

In [ ]:
_, _, mag_roberts = compute_roberts(img_camera)
_, _, mag_prewitt = compute_prewitt(img_camera)
_, _, mag_sobel, _ = compute_sobel(img_camera)

show_images(
    [
        img_camera,
        normalize_to_uint8(mag_roberts),
        normalize_to_uint8(mag_prewitt),
        normalize_to_uint8(mag_sobel),
    ],
    ["Originale", "Roberts", "Prewitt", "Sobel"],
    ncols=4,
    figsize=(16, 4)
)

## À observer
- Les trois opérateurs réagissent à peu près aux mêmes structures.
- Mais le rendu n'est pas identique.
- Sobel est souvent plus stable visuellement.
- Roberts réagit vite, parfois un peu trop vite.

## 7. Binariser la magnitude : seuillage simple
Pour obtenir une image de contours binaire, on peut appliquer un seuil sur la magnitude.

In [ ]:
def threshold_gradient(magnitude, thresh):
    """Binarise une magnitude de gradient par seuil simple.

    Paramètres
    ----------
    magnitude : numpy.ndarray
        Magnitude du gradient.
    thresh : float
        Seuil choisi manuellement.

    Retour
    ------
    numpy.ndarray
        Image binaire des contours.

    Remarques pédagogiques
    ----------------------
    - C'est la version la plus simple du passage
      “carte continue → contours binaires”.
    - Le résultat dépend fortement du seuil choisi.
    """
    binary = np.zeros_like(magnitude, dtype=np.uint8)
    binary[magnitude >= thresh] = 255
    return binary

### Démo immédiate

In [ ]:
_, _, mag_sobel_raw, _ = compute_sobel(img_camera)

thresholds = [40, 80, 120]

images = [normalize_to_uint8(mag_sobel_raw)]
titles = ["Magnitude Sobel"]

for th in thresholds:
    images.append(threshold_gradient(mag_sobel_raw, th))
    titles.append(f"Sobel + seuil {th}")

show_images(images, titles, ncols=4, figsize=(16, 4))

## À observer
- Seuil bas : beaucoup de contours, mais aussi du bruit et des détails faibles.
- Seuil élevé : résultat plus propre, mais perte de structures utiles.
- Le seuil simple est très pédagogique, mais vite limité.

## 8. Pourquoi lisser avant le gradient ?
On refait la démonstration sur une image bruitée.

### Démo directe sur image bruitée

In [ ]:
_, _, mag_clean, _ = compute_sobel(img_synth)
_, _, mag_noisy, _ = compute_sobel(img_synth_noisy)
_, _, mag_blur, _ = compute_sobel(img_synth_blur)

show_images(
    [
        img_synth,
        img_synth_noisy,
        img_synth_blur,
        normalize_to_uint8(mag_clean),
        normalize_to_uint8(mag_noisy),
        normalize_to_uint8(mag_blur)
    ],
    [
        "Image propre",
        "Image bruitée",
        "Image bruitée + flou",
        "Gradient propre",
        "Gradient sur image bruitée",
        "Gradient après flou"
    ],
    ncols=3,
    figsize=(14, 8)
)

## À observer
- Le bruit crée de faux gradients.
- Le flou gaussien ne supprime pas tout, mais il réduit fortement ces faux signaux.
- C'est exactement la logique de départ de Canny.

## 9. Laplacien, LoG et DoG
Ici, on fait un rappel minimal mais visuel.

- **Laplacien** : dérivée seconde, sensible au bruit.
- **LoG** : on lisse d'abord, puis on applique le Laplacien.
- **DoG** : différence de deux flous gaussiens, souvent utilisée comme approximation pratique du LoG.

In [ ]:
def compute_laplacian(image):
    """Calcule une réponse Laplacienne simple.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.

    Retour
    ------
    numpy.ndarray
        Réponse du Laplacien.

    Remarques pédagogiques
    ----------------------
    - Le Laplacien est une dérivée seconde.
    - Il est très sensible au bruit si on l'applique directement.
    """
    image_f = image.astype(np.float32)
    lap = cv2.Laplacian(image_f, cv2.CV_32F, ksize=3)
    return lap

In [ ]:
def compute_log(image, sigma=1.2):
    """Calcule une réponse LoG simple.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.
    sigma : float
        Paramètre du lissage gaussien.

    Retour
    ------
    numpy.ndarray
        Réponse LoG.

    Remarques pédagogiques
    ----------------------
    - LoG = lissage gaussien puis Laplacien.
    - L'idée est de dériver une image un peu plus calme.
    """
    image_f = image.astype(np.float32)
    blurred = cv2.GaussianBlur(image_f, (0, 0), sigmaX=sigma)
    log_img = cv2.Laplacian(blurred, cv2.CV_32F, ksize=3)
    return log_img

In [ ]:
def compute_dog(image, sigma1=1.0, sigma2=1.6):
    """Calcule une différence de gaussiennes.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.
    sigma1 : float
        Petite échelle.
    sigma2 : float
        Grande échelle.

    Retour
    ------
    numpy.ndarray
        Réponse DoG.

    Remarques pédagogiques
    ----------------------
    - DoG = G_sigma1 * I - G_sigma2 * I
    - Quand sigma2 est un peu plus grand que sigma1,
      DoG donne une approximation pratique du LoG.
    """
    image_f = image.astype(np.float32)
    blur1 = cv2.GaussianBlur(image_f, (0, 0), sigmaX=sigma1)
    blur2 = cv2.GaussianBlur(image_f, (0, 0), sigmaX=sigma2)
    return blur1 - blur2

### Démo immédiate

In [ ]:
lap = compute_laplacian(img_camera)
log_img = compute_log(img_camera, sigma=1.2)
dog = compute_dog(img_camera, sigma1=1.0, sigma2=1.6)

show_images(
    [img_camera, normalize_to_uint8(np.abs(lap)), normalize_to_uint8(np.abs(log_img)), normalize_to_uint8(np.abs(dog))],
    ["Originale", "|Laplacien|", "|LoG|", "|DoG|"],
    ncols=4,
    figsize=(16, 4)
)

## À observer
- Le Laplacien réagit fortement aux petites variations.
- LoG est en général plus lisible car l'image a été calmée avant dérivation.
- DoG donne une réponse voisine, souvent suffisante pour l'intuition.

## 10. Canny : un détecteur plus robuste
Rappel de la logique :

1. lissage gaussien,  
2. gradient,  
3. suppression des non-maxima,  
4. hystérésis.

Ici, OpenCV fait ces étapes pour nous.

In [ ]:
def apply_canny(image, low_thresh, high_thresh):
    """Applique le détecteur de contours de Canny.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.
    low_thresh : int
        Seuil bas de l'hystérésis.
    high_thresh : int
        Seuil haut de l'hystérésis.

    Retour
    ------
    numpy.ndarray
        Image binaire des contours.

    Remarques pédagogiques
    ----------------------
    - Les pixels très forts sont gardés.
    - Les pixels intermédiaires sont gardés seulement
      s'ils sont reliés à des contours forts.
    - C'est ce qui rend Canny plus cohérent qu'un seuil simple.
    """
    return cv2.Canny(image, threshold1=low_thresh, threshold2=high_thresh)

### Démo immédiate

In [ ]:
edges_canny = apply_canny(img_camera, 50, 120)

show_images(
    [img_camera, edges_canny],
    ["Image originale", "Contours Canny"],
    ncols=2,
    figsize=(12, 4)
)

## À observer
- Les contours sont plus fins qu'avec un simple seuil sur Sobel.
- Le résultat est souvent plus continu.
- Canny est plus proche d'un outil de pipeline que d'un simple exercice de dérivation.

## 11. Effet des paramètres de Canny
On fait varier les seuils d'hystérésis.

### Démo immédiate

In [ ]:
param_sets = [(30, 80), (50, 120), (80, 180)]

images = [img_camera]
titles = ["Originale"]

for low_t, high_t in param_sets:
    images.append(apply_canny(img_camera, low_t, high_t))
    titles.append(f"Canny\nlow={low_t}, high={high_t}")

show_images(images, titles, ncols=4, figsize=(16, 4))

## À observer
- Seuils faibles : plus de contours détectés, donc aussi plus de détails parasites.
- Seuils forts : résultat plus propre, mais certains contours deviennent incomplets.
- Il n'existe pas de seuil magique universel.

## 12. Sobel + seuil simple versus Canny
Même objectif, deux philosophies :
- **Sobel + seuil** : simple, très lisible.
- **Canny** : plus robuste, plus structuré.

### Démo immédiate

In [ ]:
sobel_binary = threshold_gradient(mag_sobel_raw, 90)
canny_binary = apply_canny(img_camera, 50, 120)

show_images(
    [img_camera, sobel_binary, canny_binary],
    ["Originale", "Sobel + seuil", "Canny"],
    ncols=3,
    figsize=(15, 4)
)

## À observer
- Sobel + seuil est idéal pour comprendre.
- Canny est souvent meilleur pour produire un résultat exploitable.
- L'hystérésis est une différence qualitative majeure.

## 13. Contour ≠ segmentation
C'est un point central.

- **Contour** : frontière locale probable.
- **Segmentation** : découpage de l'image en régions ou objets.

Une image de contours ne suffit pas forcément pour compter ou mesurer des objets.

## 14. Segmentation par seuil fixe
On commence par la méthode la plus simple.

In [ ]:
def simple_threshold(image, thresh):
    """Segmente une image par seuil fixe.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.
    thresh : int
        Seuil choisi manuellement.

    Retour
    ------
    numpy.ndarray
        Image binaire segmentée.

    Remarques pédagogiques
    ----------------------
    - Très simple.
    - Très dépendante du choix du seuil.
    - Utile pour comprendre, fragile en pratique.
    """
    _, binary = cv2.threshold(image, thresh, 255, cv2.THRESH_BINARY)
    return binary

### Démo immédiate

In [ ]:
thresholds = [80, 110, 140]

images = [img_coins]
titles = ["Image originale"]

for th in thresholds:
    images.append(simple_threshold(img_coins, th))
    titles.append(f"Seuil fixe = {th}")

show_images(images, titles, ncols=4, figsize=(16, 4))

## À observer
- Le résultat change fortement avec le seuil.
- Un seuil fixe peut suffire sur une image simple.
- Dès que l'éclairage varie, cette méthode souffre vite.

## 15. Segmentation automatique d'Otsu
Otsu cherche automatiquement un seuil global qui sépare au mieux deux classes.

In [ ]:
def otsu_threshold(image):
    """Applique le seuillage automatique d'Otsu.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.

    Retour
    ------
    otsu_value : float
        Seuil trouvé automatiquement.
    binary : numpy.ndarray
        Image binaire segmentée.

    Remarques pédagogiques
    ----------------------
    - Otsu est très pratique quand l'histogramme est assez bimodal.
    - C'est un seuil global unique.
    - Si l'éclairage est très inhomogène, Otsu peut échouer.
    """
    otsu_value, binary = cv2.threshold(
        image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    return otsu_value, binary

### Démo immédiate

In [ ]:
otsu_value, otsu_binary = otsu_threshold(img_coins)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img_coins, cmap="gray")
axes[0].set_title("Image originale")
axes[0].axis("off")

axes[1].hist(img_coins.ravel(), bins=32, color="gray", edgecolor="black")
axes[1].axvline(otsu_value, linestyle="--")
axes[1].set_title(f"Histogramme\nSeuil Otsu = {otsu_value:.1f}")

axes[2].imshow(otsu_binary, cmap="gray")
axes[2].set_title("Segmentation Otsu")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## À observer
- Otsu choisit un seuil sans intervention manuelle.
- C'est pratique, mais ce n'est pas de la magie.
- Quand fond et objets se recouvrent trop en intensité, le résultat se dégrade.

## 16. Watershed avec marqueurs
Image mentale :
- l'image devient un relief ;
- les minima deviennent des bassins ;
- les frontières entre bassins donnent les séparations.

En pratique, on n'applique presque jamais watershed “brut”.  
On prépare des **marqueurs**.

In [ ]:
def apply_watershed(image):
    """Applique un watershed guidé par des marqueurs simples.

    Paramètres
    ----------
    image : numpy.ndarray
        Image 2D en niveaux de gris.

    Retour
    ------
    color_result : numpy.ndarray
        Image couleur avec frontières en rouge.
    markers : numpy.ndarray
        Carte finale des marqueurs.
    sure_fg : numpy.ndarray
        Zone de premier plan sûr.
    unknown : numpy.ndarray
        Zone d'incertitude.

    Remarques pédagogiques
    ----------------------
    - On combine ici plusieurs idées :
      flou, seuillage, morphologie légère, distance transform.
    - Le résultat dépend fortement de la qualité des marqueurs.
    - Watershed est puissant, mais capricieux.
    """
    blurred = cv2.GaussianBlur(image, (5, 5), 1.0)

    _, thresh = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

    sure_bg = cv2.dilate(opening, kernel, iterations=2)

    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)

    unknown = cv2.subtract(sure_bg, sure_fg)

    num_labels, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(color, markers)

    color_result = color.copy()
    color_result[markers == -1] = [255, 0, 0]

    return color_result, markers, sure_fg, unknown

### Démo immédiate

In [ ]:
watershed_result, watershed_markers, sure_fg, unknown_zone = apply_watershed(img_coins)

show_images(
    [img_coins, sure_fg, unknown_zone, watershed_result],
    ["Originale", "Premier plan sûr", "Zone d'incertitude", "Watershed (frontières rouges)"],
    ncols=2,
    figsize=(12, 8)
)

## À observer
- Watershed ne produit pas juste un contour : il construit un découpage en régions.
- La qualité dépend des marqueurs.
- Sans préparation, watershed aime sur-segmenter. C'est sa passion. Presque un hobby.

## 17. Pipeline court : prétraitement → contours
On construit une chaîne simple et cohérente avec les séances 2 et 3.

### Démo immédiate

In [ ]:
img_for_edges = gaussian_blur(img_camera, ksize=5, sigma=1.0)
edges_pipeline = apply_canny(img_for_edges, 50, 120)

show_images(
    [img_camera, img_for_edges, edges_pipeline],
    ["Originale", "Prétraitement : flou gaussien", "Contours : Canny"],
    ncols=3,
    figsize=(15, 4)
)

## À observer
- Le pipeline est lisible.
- Chaque étape prépare la suivante.
- C'est exactement ce qu'on veut faire comprendre aux étudiants : on ne lance pas des algorithmes au hasard.

## 18. Pipeline court : comparer des segmentations
On met côte à côte :
- seuil fixe,
- Otsu,
- watershed.

### Démo immédiate

In [ ]:
fixed_binary = simple_threshold(img_coins, 110)
otsu_value, otsu_binary = otsu_threshold(img_coins)
watershed_result, _, _, _ = apply_watershed(img_coins)

show_images(
    [img_coins, fixed_binary, otsu_binary, watershed_result],
    ["Originale", "Seuil fixe", "Otsu", "Watershed"],
    ncols=4,
    figsize=(18, 4)
)

## À observer
- Le seuil fixe dépend de la main humaine.
- Otsu automatise ce choix, mais reste global.
- Watershed change de logique : on passe à une approche région.
- Ces méthodes sont complémentaires, pas interchangeables.

## 19. Mini-bilan
À ce stade, vous devez être capables de dire :

- ce qu'est un gradient ;
- pourquoi Sobel est souvent préféré à Roberts ;
- pourquoi on lisse avant de dériver ;
- pourquoi Canny est plus robuste qu'un simple seuillage ;
- pourquoi un contour n'est pas une segmentation ;
- dans quel cas utiliser seuil fixe, Otsu ou watershed.

## 20. Exercices guidés pour le TP
Répondez directement dans le notebook.

### Exercice 1
Sur l'image `camera`, comparer Sobel + seuil 40, 80 et 120.  
Quelle valeur vous semble le meilleur compromis ? Justifier en 2 phrases.

### Exercice 2
Sur l'image synthétique bruitée, comparer le gradient :
- sans flou gaussien,
- avec flou gaussien.

Expliquer ce qui change.

### Exercice 3
Sur `coins`, comparer :
- seuil fixe à 100,
- seuil fixe à 130,
- Otsu.

Quelle méthode vous semble la plus robuste ici ? Pourquoi ?

### Exercice 4
Relancer watershed en modifiant le coefficient du seuil de distance :
- 0.3 × `dist.max()`
- 0.5 × `dist.max()`

Observer l'effet sur les régions obtenues.

### Exercice 5
Construire un mini pipeline complet sur une image de votre choix :
- prétraitement,
- détection de contours **ou** segmentation,
- justification des paramètres.

### Exercice 1 — Sobel + seuil sur `camera`

In [ ]:
_, _, mag_ex1, _ = compute_sobel(img_camera)
mag_ex1_u8 = normalize_to_uint8(mag_ex1)

thresholds_ex1 = [40, 80, 120]
images_ex1  = [img_camera] + [threshold_gradient(mag_ex1_u8, t) for t in thresholds_ex1]
titles_ex1  = ["Originale"] + [f"Seuil = {t}" for t in thresholds_ex1]

show_images(images_ex1, titles_ex1, ncols=4)

**Réponse :**  
Le seuil **80** offre le meilleur compromis : il détecte les contours principaux (silhouette du photographe, horizon, trépied) sans noyer le résultat dans le bruit de fond présent dès 40.  
À 120, les contours deviennent fragmentés — on perd la continuité des bords, ce qui nuit à toute exploitation ultérieure (segmentation, reconnaissance de forme).

### Exercice 2 — Gradient avec et sans flou gaussien

In [ ]:
_, _, mag_ex2_raw,  _ = compute_sobel(img_synth_noisy)
blurred_ex2           = gaussian_blur(img_synth_noisy, ksize=5, sigma=1.0)
_, _, mag_ex2_blur, _ = compute_sobel(blurred_ex2)

show_images(
    [img_synth_noisy,
     normalize_to_uint8(mag_ex2_raw),
     blurred_ex2,
     normalize_to_uint8(mag_ex2_blur)],
    ["Image bruitée",
     "Gradient sans flou",
     "Image lissée",
     "Gradient après flou"],
    ncols=4
)

**Réponse :**  
Sans flou, le gradient réagit à chaque fluctuation aléatoire du bruit gaussien : la carte de magnitude est saturée de faux contours diffus qui masquent les vraies structures géométriques du rectangle synthétique.  
Avec le flou gaussien préalable, les variations de haute fréquence dues au bruit sont atténuées ; le gradient ne répond plus qu'aux transitions d'intensité réelles et nettes, rendant les contours du rectangle clairement identifiables. Le coût est une légère perte de netteté sur les vrais bords.

### Exercice 3 — Seuil fixe vs Otsu sur `coins`

In [ ]:
seg_100 = simple_threshold(img_coins, 100)
seg_130 = simple_threshold(img_coins, 130)
otsu_val_ex3, seg_otsu_ex3 = otsu_threshold(img_coins)

print(f"Seuil Otsu automatique : {otsu_val_ex3}")

show_images(
    [img_coins, seg_100, seg_130, seg_otsu_ex3],
    ["Originale", "Seuil fixe 100", "Seuil fixe 130", f"Otsu (seuil={otsu_val_ex3})"],
    ncols=4
)

**Réponse :**  
**Otsu est la méthode la plus robuste ici.** L'image `coins` présente deux populations d'intensités bien séparées (fond clair, pièces plus sombres) — exactement le cas pour lequel Otsu est optimal : il maximise la variance inter-classes et trouve le seuil naturel sans intervention manuelle.  
Le seuil fixe à 100 sur-segmente (pixels sombres du fond inclus), celui à 130 sous-segmente ou fragmentes certaines pièces. Otsu détermine automatiquement la valeur correcte et reste robuste si l'illumination globale varie légèrement d'une image à l'autre.

### Exercice 4 — Watershed : effet du coefficient de distance

In [ ]:
def apply_watershed_coeff(image, dist_coeff):
    """Watershed paramétrable par le coefficient du seuil de distance."""
    gray  = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR) if image.ndim == 2 else image
    gray8 = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY) if gray.ndim == 3 else gray
    _, binary = cv2.threshold(gray8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Élimination du fond
    kernel     = np.ones((3, 3), np.uint8)
    sure_bg    = cv2.dilate(binary, kernel, iterations=3)
    dist       = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, dist_coeff * dist.max(), 255, 0)
    sure_fg    = np.uint8(sure_fg)
    unknown    = cv2.subtract(sure_bg, sure_fg)

    _, markers = cv2.connectedComponents(sure_fg)
    markers    = markers + 1
    markers[unknown == 255] = 0

    img_bgr = cv2.cvtColor(gray8, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(img_bgr, markers)
    result  = gray8.copy()
    result[markers == -1] = 255
    return result

ws_03 = apply_watershed_coeff(img_coins, dist_coeff=0.3)
ws_05 = apply_watershed_coeff(img_coins, dist_coeff=0.5)

show_images(
    [img_coins, ws_03, ws_05],
    ["Originale", "Watershed coeff=0.3", "Watershed coeff=0.5"],
    ncols=3
)

**Réponse :**  
- **Coefficient 0.3** : le seuil sur la transformée de distance est bas, donc les zones de premier plan certain (`sure_fg`) sont plus étendues. Les marqueurs initiaux couvrent une plus grande partie des pièces — watershed génère des régions plus larges, avec moins de sur-segmentation mais un risque de fusionner des pièces adjacentes.  
- **Coefficient 0.5** : seuil plus strict, marqueurs plus conservateurs (uniquement les cœurs éloignés des bords). Watershed produit des régions mieux séparées pour les pièces qui se chevauchent, au prix d'une plus grande zone d'incertitude et parfois d'une légère sur-segmentation des grandes pièces.

### Exercice 5 — Mini pipeline complet

In [ ]:
# Image choisie : moon (texture lunaire + bord très marqué)
img_moon_ex5 = data.moon().astype(np.uint8)

# Étape 1 — Prétraitement : flou gaussien pour atténuer le grain
img_moon_blur = gaussian_blur(img_moon_ex5, ksize=5, sigma=1.2)

# Étape 2 — Détection de contours : Canny avec hystérésis calibrée
# Seuil bas à 30 pour conserver les cratères, haut à 90 pour éviter le bruit résiduel
edges_moon = apply_canny(img_moon_blur, low_thresh=30, high_thresh=90)

# Étape 3 — Segmentation complémentaire : Otsu pour séparer zones claires / sombres
otsu_val_moon, seg_moon = otsu_threshold(img_moon_ex5)
print(f"Seuil Otsu lune : {otsu_val_moon}")

show_images(
    [img_moon_ex5, img_moon_blur, edges_moon, seg_moon],
    ["Originale", "Après flou (σ=1.2)", "Contours Canny (30/90)", f"Otsu ({otsu_val_moon})"],
    ncols=4
)

**Justification des paramètres :**  

| Étape | Choix | Raison |
|---|---|---|
| Flou gaussien `ksize=5, σ=1.2` | Lissage modéré | L'image `moon` a un grain de texture fin ; un σ trop grand efface les petits cratères qu'on veut détecter |
| Canny `low=30, high=90` | Seuils bas–modérés | Le contraste des cratères est faible ; des seuils trop élevés (> 120) fragmentent les bords circulaires |
| Otsu | Automatique | La bimodalité de l'histogramme (zones sombres/claires) rend Otsu adapté pour une segmentation globale complémentaire sans réglage manuel |

**Ce que montre le pipeline :**  
Canny s'occupe des frontières locales (cratères, bord du disque lunaire) tandis qu'Otsu distingue les grandes régions. Les deux sont complémentaires : contours ≠ segmentation.

## 21. Conclusion
Chaînes minimales à retenir :

1. **Filtrer puis détecter des contours**  
   `image → flou gaussien → Canny`

2. **Segmenter automatiquement par seuillage**  
   `image → Otsu`

3. **Segmenter par régions avec préparation**  
   `image → prétraitement → marqueurs → watershed`

Le plus important n'est pas seulement d'obtenir une image “jolie”.  
Le plus important est de pouvoir **justifier chaque étape**.